# From a BigQuery table to a CSV and to a Cloud Storage bucket, from Colab

**DAISY 2026 summer school, Taranto. Session "Publications as innovation data: measuring (green) science with OpenAlex, at scale with BigQuery"** (Massimiliano Coda Zabetta)

The SQL scripts in `sql/` leave their results as tables inside BigQuery. This notebook shows the last step, getting a table out of the warehouse and into a file you can keep: read a table into pandas straight from Colab, write it to CSV, and copy the CSV to a Google Cloud Storage **bucket** (a folder in the cloud that other machines, and your future self, can read). The example table is `daisy.q03_ce_patcit_pairs`, the article-to-patent pairs written by `sql/q03_patcit_green.sql`; any table in your `daisy` dataset works the same way.

Two placeholders to replace before running:

- `your-project-id`: the id of your BigQuery project, shown in the console's project picker (the id, not the display name).
- `your-bucket-name`: the name of a bucket you own (see the bucket section below).

Steps 1 to 4 (read, look, save to CSV) run in a free sandbox project. Step 5 (the bucket) needs a project with billing enabled: a sandbox has no Cloud Storage. If you have no bucket, stop after step 4 and download the CSV from Colab's file pane, or use the Drive alternative at the end.

## 1. Sign in and connect to your project

Colab runs on a Google machine that knows nothing about you. `authenticate_user()` opens the Google login popup; use the same account that owns the BigQuery project. The credential is valid for this Colab runtime only.

In [ ]:
from google.colab import auth
auth.authenticate_user()        # Google login popup; grant access, then the cell finishes

In [ ]:
from google.cloud import bigquery                 # the BigQuery client library, preinstalled in Colab

PROJECT_ID = "your-project-id"                    # replace: the project id from the console's project picker

client = bigquery.Client(project=PROJECT_ID)      # every query is billed to (and cached in) this project
client.query("SELECT 1").result()                 # a query that reads no table, so it costs nothing:
print("connected to", client.project)             # if this line prints, login and project id are both right

## 2. Read a table into pandas with `%%bigquery`

`%%bigquery name` at the top of a cell sends the rest of the cell to BigQuery and stores the result as a pandas DataFrame called `name`. The magic must be loaded once per runtime, and told which project pays.

`SELECT *` is a sin on `openalex_walden.works` (every column of every row is billed) but harmless here: this is our own table, a few hundred thousand rows at most, and we want all of it.

In [ ]:
%load_ext google.cloud.bigquery
# enables the %%bigquery cell magic (the comment sits on its own line: a trailing "# ..." on a %-line is passed to the magic)
from google.cloud.bigquery import magics
magics.context.project = PROJECT_ID               # the project the magic bills; same as the client above

In [ ]:
%%bigquery results
SELECT *
FROM `your-project-id.daisy.q03_ce_patcit_pairs`

## 3. Look before saving

The DataFrame is in memory now. Check the size and a few rows: the same numbers as the console's result grid.

In [ ]:
print(results.shape)                              # (rows, columns)
results.head()                                    # first five rows

## 4. Write it to a CSV on the Colab machine

`astype(str)` turns every column into text first. Without it pandas rewrites the nulls of the un-cited articles (no `publication_number`) as `NaN` and the integer columns that contain them as floats (`2015.0`), and Stata's `import delimited` then has to be told what to do with each. As text, the file reads back exactly as the table looked.

The file lands on the Colab machine's disk (the folder icon in the left pane), which is wiped when the runtime ends. Keep it: download it from the file pane, or copy it to a bucket (next step).

In [ ]:
CSV_NAME = "q03_ce_patcit_pairs.csv"

results.astype(str).to_csv(CSV_NAME, index=False)  # index=False: no extra unnamed row-number column

import os
print(CSV_NAME, round(os.path.getsize(CSV_NAME) / 1e6, 1), "MB")

## 5. Copy the CSV to a Cloud Storage bucket

A bucket is a named container in Google Cloud Storage. Files in it survive the Colab runtime, can be read from any machine that has the right (a colleague's Colab, a server, BigQuery itself with `EXPORT DATA` and `LOAD DATA`), and are the normal way to move data larger than a download.

**Creating one** (once, in the console, in a project with billing enabled): Cloud Storage > Buckets > Create. Bucket names are global, so pick something unlikely to be taken; leave the region and the storage class at their defaults. Storage costs cents per GB per month, an order of magnitude below the cost of re-running the query.

`gsutil` is the command-line tool for Cloud Storage; it is preinstalled in Colab and uses the login of step 1. The `!` in front runs a shell command from the notebook, and `{BUCKET}` inside it is substituted from the Python variable. Google now recommends `gcloud storage cp` (same arguments, also preinstalled); `gsutil` still works.

In [ ]:
BUCKET = "your-bucket-name"                       # replace: the name of your bucket (no gs://, no slash)

!gsutil cp {CSV_NAME} gs://{BUCKET}/{CSV_NAME}
!gsutil ls -l gs://{BUCKET}/
# first line: copy the file up (gs://bucket/path is a cloud file address); second: list the bucket, file size and date

**Reading it back**, from any Colab with the same login: `!gsutil cp gs://{BUCKET}/{CSV_NAME} .` copies the file down, then `pd.read_csv(CSV_NAME)`. Stata, on your own machine: download the file from the console (Cloud Storage > Buckets > your bucket > the file), then `import delimited`.

**Alternative without a bucket** (sandbox users): mount Google Drive and write there; the file then appears in Drive like any other document.

In [ ]:
# Sandbox alternative: Google Drive instead of a bucket (a permission popup on the first run).
# from google.colab import drive
# drive.mount("/content/drive")
# results.astype(str).to_csv("/content/drive/MyDrive/" + CSV_NAME, index=False)

## Pitfalls, in one place

- The Colab disk is temporary: a CSV that is not downloaded, copied to a bucket or to Drive is gone when the runtime ends.
- Sandbox tables expire after 60 days: export what you keep.
- `%%bigquery` bills the project in `magics.context.project`; check it before a big read. `SELECT *` is fine on your own small tables, not on `openalex_walden.works`.
- Bucket names are global: anyone can tell a name is taken, but the files inside stay private unless you change the bucket's permissions.
- For tables too big for pandas (many GB), skip Colab: `EXPORT DATA` in SQL writes sharded CSVs straight from BigQuery to the bucket (`sql/q04_export_examples.sql`, block 3).